In [ ]:
from pathlib import Path
from metasmith.python_api import Agent, Source, Std, DataInstanceLibrary, TransformInstanceLibrary, WorkflowTask
from metasmith.python_api import DataTypeLibrary, Endpoint
from local.constants import WORKSPACE_ROOT

# dtypes, containers, transforms = Std()

path_to_agent_home = Path("./cache/local_home").resolve()
smith = Agent(
    home = Source.FromLocal(path_to_agent_home),
)
# smith.Deploy()

In [ ]:
mock_types = DataTypeLibrary(types=dict(
    a=Endpoint({"test", "a"}),
    b=Endpoint({"test", "b"}),
    x=Endpoint({"test", "x"}),
    y=Endpoint({"test", "y"}),
))

transforms = TransformInstanceLibrary("./transforms/simple_1", include_std=False)
transforms.AddTypeLibrary("mock", mock_types)
transforms.Save() # updates types
transforms.AddStub("no_op")
transforms.AddStub("no_op_no_fail")
transforms.Save()

In [ ]:
inputs = DataInstanceLibrary("./cache/dev21.mock.xgdb")
samples = []
for i in range(2):
    in_path = WORKSPACE_ROOT/f"main/local_mock/cache/test/mock_d.{i}"
    in_path.parent.mkdir(exist_ok=True)
    with open(in_path, "w") as f:
        f.write("2")
    inputs.AddTypeLibrary("mock", mock_types)
    # inputs.AddItem(in_path, "mock::a")
    inputs.AddItem(in_path, "mock::x")
    samples.append(in_path)
inputs.Save()
for p, n, e in inputs.Iterate():
    print(n, e, e.parents)

In [ ]:
for loc, t,  in transforms.IterateTransforms():
    print(t.model)

In [ ]:
task = smith.GenerateWorkflow(
    samples    = [inputs.AsView({p}) for p in samples],
    resources  = [],
    transforms = [transforms],
    # targets    = [mock_types["b"]]
    targets    = [mock_types["y"]]
)
print(task.GetKey())

In [ ]:
smith.StageWorkflow(task, on_exist='clear', verify_external_paths=False)

In [ ]:
with open(WORKSPACE_ROOT/"secrets/slurm_account_fir") as f:
    SLURM_ACCOUNT = f.read()

# params = dict(
#     slurmAccount = SLURM_ACCOUNT,
#     executor_queueSize = 100,
#     process = dict(
#         tries=3,
#         array=5,
#         cpus=1,
#         memory='32 GB',
#         time='12hours',
#     ),
# )
params = {}
smith.RunWorkflow(task, smith.GetNxfConfigPresets()["local"], params)

In [ ]:
import re
re.match(r"^sample\s\d+,\s?step\s\d+$", "sample 1, step 1")